In [4]:
# Standard library
from pathlib import Path

# Third-party libraries
import joblib
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC


In [5]:
"""
Configuration for training the CV + Linear SVM model.

This module defines:
- Project paths
- Dataset and model locations
- Hyperparameter grid for model selection
"""

# Data
try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    # fallback for notebooks or interactive environments
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR: Path = PROJECT_ROOT / "data" / "interim"
MODEL_DIR: Path = PROJECT_ROOT / "models" / "experiment"

DATA_FILE: str = "WELFake_Dataset.csv"
DATA_PATH: Path = DATA_DIR / DATA_FILE
MODEL_FILE: Path = MODEL_DIR / "best_cv_svm_model.pkl"

# Model hyperparameters
PARAM_GRID = [
    {   
        # Larger C => weaker regularization
        "C": [0.1, 1, 10],

        # Adjust weights to handle class imbalance
        "class_weight": [None, "balanced"],

        # Squared hinge is smoother and typically more stable
        "loss": ["squared_hinge"],

        # Dual formulation is recommended for LinearSVC
        "dual": [True],

        # Increase iterations to avoid convergence warnings
        "max_iter": [5000],
    }
]


In [6]:
def running_in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


def identity(x):
    return x

In [7]:
"""
 CV + Linear SVM Training

This notebook trains and evaluates a CV + Linear SVM classifier using
preprocessed train/test splits. Hyperparameters are optimized via GridSearchCV.
"""

X_train = joblib.load(DATA_DIR / "X_train.pkl")
y_train = joblib.load(DATA_DIR / "y_train.pkl")

X_test  = joblib.load(DATA_DIR / "X_test.pkl")
y_test  = joblib.load(DATA_DIR / "y_test.pkl")

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")



Train size: 57229
Test size:  14308


In [8]:

cv = CountVectorizer(
    
    tokenizer=identity,     
    preprocessor=identity,   
    token_pattern=None,
)

cv.fit(X_train)  # freeze on full training set
X_train_cv = cv.transform(X_train)
X_test_cv = cv.transform(X_test)

N_JOBS = 1 if running_in_notebook() else -1

grid = GridSearchCV(
    LinearSVC(),
    param_grid=PARAM_GRID,
    cv=5,
    scoring="accuracy",
    verbose=3,
    n_jobs=-1
)
grid.fit(X_train_cv, y_train)

print("Beste parameters:", grid.best_params_)
print("Beste CV-score:", grid.best_score_)

# Apply the best grid searched hyperparameters for the LSVC
y_pred = grid.predict(X_test_cv)

# Evaluate
test_accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", test_accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


# Save predictions
pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred
})

# Save trained model
joblib.dump(grid.best_estimator_, MODEL_FILE)


Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/pietervanbrakel/zelfstudie/projects/scriptie/.venv/lib/python3.9/site-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/pietervanbrakel/zelfstudie/projects/scriptie/.venv/lib/python3.9/site-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 1/5] END C=1, class_weight=None, dual=True, loss=squared_hinge, max_iter=5000;, score=0.958 total time=  56.3s
[CV 2/5] END C=1, class_weight=None, dual=True, loss=squared_hinge, max_iter=5000;, score=0.961 total time=  56.3s


/Users/pietervanbrakel/zelfstudie/projects/scriptie/.venv/lib/python3.9/site-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 2/5] END C=0.1, class_weight=None, dual=True, loss=squared_hinge, max_iter=5000;, score=0.966 total time=  58.9s


...

[CV 1/5] END C=10, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.956 total time=  30.3s
[CV 4/5] END C=10, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.959 total time=  28.6s


/Users/pietervanbrakel/zelfstudie/projects/scriptie/.venv/lib/python3.9/site-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 3/5] END C=10, class_weight=balanced, dual=True, loss=squared_hinge, max_iter=5000;, score=0.953 total time=  29.1s
Beste parameters: {'C': 0.1, 'class_weight': 'balanced', 'dual': True, 'loss': 'squared_hinge', 'max_iter': 5000}
Beste CV-score: 0.9641789955333409
Test Accuracy: 0.9647050601062342

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.96      0.96      7081
           1       0.96      0.97      0.97      7227

    accuracy                           0.96     14308
   macro avg       0.96      0.96      0.96     14308
weighted avg       0.96      0.96      0.96     14308


Confusion Matrix:
 [[6790  291]
 [ 214 7013]]


/Users/pietervanbrakel/zelfstudie/projects/scriptie/.venv/lib/python3.9/site-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


['/Users/pietervanbrakel/zelfstudie/projects/scriptie/models/experiment/best_cv_svm_model.pkl']

In [9]:
frequencies = sum(X_train_cv).toarray()[0]
ngrams = pd.DataFrame(frequencies, index = cv.get_feature_names_out(), columns = ['frequency'])
ngrams = ngrams.sort_values(by='frequency', ascending = False)
ngrams.iloc[:50]

,frequency
’,296874
“,210670
”,209283
said,184805
trump,165435
state,88058
u,85420
would,84353
people,71988
one,70865
